# 02 — Cleaning & Feature Engineering

Reads `master_long.parquet`, cleans the signal, detects anomalies, and writes:

| Output | Description |
|--------|-------------|
| `master_long_clean.parquet` | Granular 1 Hz, cleaned, with anomaly flags |
| `master_short.parquet` | One row per participant × day, aggregated features |

In [1]:
import sys, logging
from pathlib import Path

import pandas as pd
import numpy as np
import yaml

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.feature_engineering import (
    remove_hr_outliers,
    detect_time_gaps,
    interpolate_short_gaps,
    add_anomaly_flags,
    build_master_short,
)

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
CONFIG_PATH   = REPO_ROOT / 'config' / 'participants.yaml'

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger('02_cleaning')

In [2]:
# ── Load master_long ──────────────────────────────────────────────────────────
long_path = PROCESSED_DIR / 'master_long.parquet'

if not long_path.exists():
    raise FileNotFoundError(
        f'{long_path} not found.\n'
        'Run notebook 01_data_ingestion.ipynb first.'
    )

df = pd.read_parquet(long_path, engine='pyarrow')
print(f'Loaded master_long: {df.shape}')
df.head(3)

Loaded master_long: (21, 11)


,Time,HR_bpm,Time_s,User_ID,Date,Day_of_Week,session_id,hrv_source,height_cm,weight_kg,duration_meta
0,0 days 00:00:00,80.0,0,polar01,2026-06-18,Thursday,d9e152be-f4ca-4e76-baf5-badb731e67f3,rr_intervals_ble,178,72,20
1,0 days 00:00:01,77.8,1,polar01,2026-06-18,Thursday,d9e152be-f4ca-4e76-baf5-badb731e67f3,rr_intervals_ble,178,72,20
2,0 days 00:00:02,78.9,2,polar01,2026-06-18,Thursday,d9e152be-f4ca-4e76-baf5-badb731e67f3,rr_intervals_ble,178,72,20


In [3]:
# ── Step 1 — Remove physiologically impossible HR ─────────────────────────────
n_before = df['HR_bpm'].notna().sum()
df = remove_hr_outliers(df)
n_after  = df['HR_bpm'].notna().sum()
removed = n_before - n_after
print(f'Outlier HR removed : {removed:,} rows ({100*removed/len(df):.2f}%)')

Outlier HR removed : 0 rows (0.00%)


In [4]:
# ── Step 2 — Detect and mark time gaps ───────────────────────────────────────
df = df.groupby(['User_ID', 'Date'], group_keys=False).apply(detect_time_gaps)
n_gaps = df['gap_before'].sum()
print(f'Gap events detected: {n_gaps:,}')

Gap events detected: 0


/tmp/ipykernel_662661/488528554.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['User_ID', 'Date'], group_keys=False).apply(detect_time_gaps)


In [5]:
# ── Step 3 — Interpolate short gaps (≤ 5 s) ──────────────────────────────────
nan_before = df['HR_bpm'].isna().sum()
df = df.groupby(['User_ID', 'Date'], group_keys=False).apply(interpolate_short_gaps)
nan_after  = df['HR_bpm'].isna().sum()
filled = nan_before - nan_after
print(f'NaN rows filled by interpolation: {filled:,}')
print(f'Remaining NaN HR: {nan_after:,}')

NaN rows filled by interpolation: 0
Remaining NaN HR: 0


/tmp/ipykernel_662661/880089847.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['User_ID', 'Date'], group_keys=False).apply(interpolate_short_gaps)


In [6]:
# ── Step 4 — Anomaly detection ────────────────────────────────────────────────
df = add_anomaly_flags(df, z_thresh=3.0, use_iqr=True)

n_anom = df['is_anomaly'].sum()
print(f'Anomalous rows (z-score OR IQR): {n_anom:,}  ({100*n_anom/len(df):.2f}%)')

Anomalous rows (z-score OR IQR): 3  (14.29%)


In [7]:
# ── Save master_long_clean ────────────────────────────────────────────────────
clean_path = PROCESSED_DIR / 'master_long_clean.parquet'
df.to_parquet(clean_path, index=False, engine='pyarrow')
print(f'Saved → {clean_path.relative_to(REPO_ROOT)}')
print(f'Shape  : {df.shape}')
df.dtypes

Saved → data/processed/master_long_clean.parquet
Shape  : (21, 15)


Time                 timedelta64[ns]
HR_bpm                       float64
Time_s                         int64
User_ID                       object
Date                  datetime64[ns]
Day_of_Week                   object
session_id                    object
hrv_source                    object
height_cm                      int64
weight_kg                      int64
duration_meta                  int64
gap_before                      bool
is_anomaly_zscore               bool
is_anomaly_iqr                  bool
is_anomaly                      bool
dtype: object

In [8]:
# ── Load participant demographics from config ─────────────────────────────────
with open(CONFIG_PATH, encoding='utf-8') as fh:
    config = yaml.safe_load(fh)

demo_rows = []
for p in config.get('participants', []):
    demo_rows.append({
        'User_ID'   : p['folder'],
        'p_age'     : p.get('age'),
        'p_sex'     : p.get('sex'),
        'p_height'  : p.get('height_cm'),
        'p_weight'  : p.get('weight_kg'),
        'p_hr_max'  : p.get('hr_max'),
        'p_hr_rest' : p.get('hr_rest'),
        'p_vo2max'  : p.get('vo2max'),
    })

demo_df = pd.DataFrame(demo_rows)
print(demo_df)

   User_ID p_age p_sex  p_height  p_weight p_hr_max p_hr_rest p_vo2max
0  polar01  None  None     178.0      72.0     None      None     None
1  polar02  None  None     186.0     100.0     None      None     None
2  polar03  None  None     160.0      55.0     None      None     None


In [9]:
# ── Build and save master_short ───────────────────────────────────────────────
short = build_master_short(df, meta_df=demo_df)

short_path = PROCESSED_DIR / 'master_short.parquet'
short.to_parquet(short_path, index=False, engine='pyarrow')
print(f'Saved → {short_path.relative_to(REPO_ROOT)}')
print(f'Shape  : {short.shape}')
short.head()

Saved → data/processed/master_short.parquet
Shape  : (1, 22)


/home/bruno1008/Neroes/neroes_polar_pipeline/src/feature_engineering.py:172: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  short = df_long.groupby(agg_key).apply(_agg).reset_index()


,User_ID,Date,FC_mean,FC_std,FC_variance,FC_min,FC_max,FC_range,HR_resting_mean,lnRMSSD,...,data_quality_pct,calories,Day_of_Week,p_age,p_sex,p_height,p_weight,p_hr_max,p_hr_rest,p_vo2max
0,polar01,2026-06-18,82.990476,5.69578,32.441905,75.5,97.5,22.0,82.990476,3.501,...,100.0,NaN,Thursday,None,None,178.0,72.0,None,None,None


In [10]:
# ── Quality summary ───────────────────────────────────────────────────────────
print('=== Data quality per participant ===')
quality = (
    df.groupby('User_ID')
    .apply(lambda g: pd.Series({
        'total_rows'    : len(g),
        'valid_hr_pct'  : round(100 * g['HR_bpm'].notna().mean(), 2),
        'anomaly_pct'   : round(100 * g['is_anomaly'].mean(), 2),
        'sessions'      : g['Date'].nunique(),
    }))
    .reset_index()
)
print(quality.to_string(index=False))

=== Data quality per participant ===
User_ID  total_rows  valid_hr_pct  anomaly_pct  sessions
polar01        21.0         100.0        14.29       1.0


/tmp/ipykernel_662661/3521248187.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
